# Sunspot Analysis & Prediction (Colab Version)

This notebook uses the professional `src` modules to load data, engineer features, and train a hybrid Ridge + XGBoost + EVT model.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# --- Environment Setup ---
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    # UPDATE THIS PATH if needed
    project_path = '/content/drive/My Drive/Sunspots' 
    if project_path not in sys.path:
        sys.path.append(project_path)
    os.chdir(project_path)
    print(f"Running in Colab. Directory: {os.getcwd()}")
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        sys.path.append(os.path.abspath('..'))
        os.chdir('..')
    print(f"Running locally. Directory: {os.getcwd()}")

from src.utils import load_config, plot_sunspots_series, plot_predictions
from src.data import load_data
from src.features import create_features, prepare_target
from src.train import train_evaluate, run_future_forecast

## 1. Load Data
Negative values (like -1) are automatically corrected to 0 in `src/data.py`.

In [ ]:
config = load_config('config.yaml')
df = load_data(config['data']['url'], save_path=config['data']['save_path'])
df.head()

## 2. Define Evaluation Period
Choose the range of data to use for features and evaluation.

In [ ]:
# Define your period here
start_period = '1965-01-01'  # Default: 1965
end_period = df.index.max().strftime('%Y-%m-%d')

df_filtered = df.loc[start_period:end_period].copy()
print(f"Data points in period: {len(df_filtered)}")
plot_sunspots_series(df_filtered, title=f"Sunspots from {start_period} to {end_period}")

## 3. Training & Evaluation
Using Walk-Forward Validation.

In [ ]:
# Prepare features
df_feat = create_features(df_filtered, config['features']['lags'], config['features']['rolling_windows'])
df_feat = prepare_target(df_feat, shift=config['features']['target_shift'])

X = df_feat.drop(columns=['target', 'SUNSPOTS', 'LOG_SUNSPOTS'])
y = df_feat['target']

results = train_evaluate(X, y, config)

print(f"Evaluation results for period {start_period} to {end_period}:")
print(f"- Hybrid RMSE: {results['hybrid_rmse']:.4f}")
print(f"- Hybrid MAE:  {results['hybrid_mae']:.4f}")

## 4. Visualizing Predictions vs Real
This compares the walk-forward predictions against the actual values.

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(results['actuals'], label='Actual', alpha=0.6, color='blue')
plt.plot(results['predictions'], label='Forecast', alpha=0.8, color='red', linestyle='--')
plt.title("Walk-Forward Forecast vs Real Values")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 5. Future Forecasting
Make a prediction for multiple days ahead.

In [ ]:
# Steps to forecast
steps_ahead = 5 

# Get predictions for each day
future_predictions = run_future_forecast(df_filtered, steps=steps_ahead, config=config)

print("date num of sunspots")
for date, val in future_predictions.items():
    print(f"{date.strftime('%d/%m/%Y')} {val:.2f}")

## 6. Export Models
We save the configuration and the raw filtered data so the Gradio app can reconstruct features and models on the fly (since we need to refit for each specific horizon).

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(df_filtered, 'models/sunspots_data.joblib')
joblib.dump(config, 'models/config.joblib')
joblib.dump(results, 'models/validation_results.joblib')
print("Model data, config, and validation results exported to /models")